# Notebook 02 — CNN Training

This notebook trains the custom `RetinalCNN` end-to-end and analyses its performance.
All hyperparameters live in a single config dict so you can tweak without hunting through code.

**Outline:**
1. Config + model summary
2. Full 20-epoch training loop with live curves
3. Per-class accuracy analysis
4. Prediction grid (3 images/class, colour-coded correct/wrong)
5. Save checkpoint

## Cell 1 — Config

In [ ]:
import sys, os
sys.path.insert(0, '..')
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import random
from IPython.display import clear_output
from PIL import Image

plt.rcParams['figure.dpi'] = 110
torch.manual_seed(42)

# ── All hyperparameters as named constants in one dict ─────────────────────
CONFIG = dict(
    lr          = 1e-3,        # Adam initial learning rate
    lr_step     = 10,           # halve LR every this many epochs (StepLR)
    lr_gamma    = 0.5,         # LR decay factor
    batch_size  = 32,          # mini-batch size
    epochs      = 20,          # total training epochs
    num_classes = 4,           # CNV / DME / DRUSEN / NORMAL
    hidden_dim  = 128,         # CNN final conv filter count (= Transformer d_model)
    weight_decay= 1e-4,        # L2 regularisation on weights
    num_workers = 4,           # DataLoader workers
    data_dir    = '../data',   # path to dataset root
    results_dir = '../results',
)
os.makedirs(CONFIG['results_dir'], exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:<15} = {v}")

## Cell 2 — Model Summary

We print a layer-by-layer summary showing tensor shapes at each stage.

### Why GAP produces `(128,)` not `(128×7×7,)`

After `block3`, feature maps are `[B, 128, 28, 28]`.  
- **Flatten** would concatenate all 28×28 = 784 spatial positions per channel → `[B, 128×784]` = `[B, 100352]`.
  This works for classification but **destroys the spatial map** that Grad-CAM needs.
- **Global Average Pooling (GAP)** computes `mean` over the `(28, 28)` spatial grid for each of the 128 channels → `[B, 128]`.
  The 2D activation map `A^k` **still exists** as a hook-accessible tensor before pooling,
  enabling Grad-CAM to upsample it back to 224×224.

In [ ]:
from src.model_cnn import RetinalCNN
from torchsummary import summary

model = RetinalCNN(num_classes=CONFIG['num_classes']).to(DEVICE)
print("\n=== RetinalCNN Layer Summary ===")
summary(model, input_size=(3, 224, 224), device=str(DEVICE))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {n_params:,}")

## Cell 3 — Training Loop (20 epochs) with Live Curves

In [ ]:
from src.dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=CONFIG['data_dir'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

optimizer = Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = StepLR(optimizer, step_size=CONFIG['lr_step'], gamma=CONFIG['lr_gamma'])
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

def run_epoch(loader, training=True):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, labels)
            if training:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += images.size(0)
    return total_loss / total, correct / total

print(f"Training for {CONFIG['epochs']} epochs…\n")
for epoch in range(1, CONFIG['epochs'] + 1):
    tl, ta = run_epoch(train_loader, training=True)
    vl, va = run_epoch(val_loader,   training=False)
    scheduler.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    if va > best_val_acc:
        best_val_acc = va
        torch.save({'epoch': epoch, 'model_name': 'cnn',
                    'state_dict': model.state_dict(), 'val_acc': va},
                   f"{CONFIG['results_dir']}/best_cnn.pth")
        marker = ' ✓'
    else:
        marker = ''
    print(f"Epoch {epoch:>2}/{CONFIG['epochs']}  "
          f"train_loss={tl:.4f}  train_acc={ta*100:.1f}%  "
          f"val_loss={vl:.4f}  val_acc={va*100:.1f}%{marker}")

# ── Plot curves ────────────────────────────────────────────────────────────
eps = range(1, CONFIG['epochs'] + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(eps, history['train_loss'], label='Train', lw=2)
ax1.plot(eps, history['val_loss'],   label='Val',   lw=2, ls='--')
ax1.set(title='Loss', xlabel='Epoch', ylabel='Cross-Entropy'); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(eps, [a*100 for a in history['train_acc']], label='Train', lw=2)
ax2.plot(eps, [a*100 for a in history['val_acc']],   label='Val',   lw=2, ls='--')
ax2.set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy (%)'); ax2.legend(); ax2.grid(alpha=.3)
plt.suptitle('RetinalCNN Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/cnn_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\nBest val accuracy: {best_val_acc*100:.2f}%")

## Cell 4 — Per-Class Accuracy Analysis

Overall accuracy obscures class-level failures. We break it down here.

**Expected pattern:** DRUSEN and DME tend to be the hardest classes.
- **DRUSEN** images show subtle bumpy irregularities in the RPE that can look similar to a mild DME pattern.
- **DME** fluid pockets can overlap visually with CNV subretinal fluid — the distinction lies in the *location* of the fluid (intraretinal vs subretinal), which a CNN without explicit attention may partially confuse.
- **CNV** and **NORMAL** are typically easier because their OCT signatures are highly distinctive.

In [ ]:
# Per-class accuracy on the test set
model.eval()
class_correct = {c: 0 for c in CLASS_NAMES}
class_total   = {c: 0 for c in CLASS_NAMES}

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(1)
        for pred, lbl in zip(preds, labels):
            cls = CLASS_NAMES[lbl.item()]
            class_correct[cls] += (pred == lbl).item()
            class_total[cls]   += 1

accs = {c: class_correct[c] / class_total[c] * 100 for c in CLASS_NAMES}
print("Per-class test accuracy:")
for cls, acc in accs.items():
    bar = '█' * int(acc / 2)
    print(f"  {cls:<10}  {acc:5.1f}%  {bar}")

colors = ['#e63946' if v == min(accs.values()) else '#457b9d' for v in accs.values()]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(accs.keys(), accs.values(), color=colors, width=0.6, edgecolor='white')
for bar, val in zip(bars, accs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold')
ax.set_ylim(0, 110)
ax.axhline(90, ls='--', color='gray', alpha=.5, label='90% line')
ax.set(title='Per-Class Test Accuracy — RetinalCNN', ylabel='Accuracy (%)')
ax.legend(); plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/perclass_accuracy.png", dpi=150, bbox_inches='tight')
plt.show()

## Cell 5 — Prediction Grid (3 per class, colour-coded)

Green border = correct prediction. Red border = wrong prediction. Confidence shown in title.

In [ ]:
import torchvision.transforms.functional as TF

N_PER_CLASS = 3
# Collect sample images from test set by class
class_samples = {c: [] for c in CLASS_NAMES}

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        imgs_dev = images.to(DEVICE)
        logits   = model(imgs_dev)
        probs    = torch.softmax(logits, dim=1).cpu()
        preds    = logits.argmax(1).cpu()
        for img, lbl, pred, prob in zip(images, labels, preds, probs):
            cls = CLASS_NAMES[lbl.item()]
            if len(class_samples[cls]) < N_PER_CLASS:
                class_samples[cls].append({'img': img, 'true': lbl.item(),
                                            'pred': pred.item(), 'conf': prob[pred].item()})
        if all(len(v) == N_PER_CLASS for v in class_samples.values()):
            break

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])
def denorm(t):
    img = t.permute(1,2,0).numpy()
    return np.clip(img * STD + MEAN, 0, 1)

fig, axes = plt.subplots(4, N_PER_CLASS, figsize=(N_PER_CLASS*3.5, 16))
fig.suptitle('Prediction Grid — RetinalCNN (green=correct, red=wrong)', fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    for col, sample in enumerate(class_samples[cls]):
        ax = axes[row, col]
        ax.imshow(denorm(sample['img']))
        correct = sample['true'] == sample['pred']
        color   = '#2dc653' if correct else '#e63946'
        for sp in ax.spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(4); sp.set_visible(True)
        pred_name = CLASS_NAMES[sample['pred']]
        ax.set_title(f"True: {cls}\nPred: {pred_name}  ({sample['conf']*100:.0f}%)",
                     fontsize=9, color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/prediction_grid.png", dpi=150, bbox_inches='tight')
plt.show()

## Cell 6 — Save Model & Final Test Accuracy

The checkpoint saved here is loaded by notebooks 03 and 04.
It contains the epoch, val accuracy, model name, and full `state_dict`.

In [ ]:
# Final test-set accuracy using best checkpoint
ckpt = torch.load(f"{CONFIG['results_dir']}/best_cnn.pth", map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt['state_dict'])
model.eval()

correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

test_acc = correct / total
print(f"Final test accuracy (best checkpoint, epoch {ckpt['epoch']}): {test_acc*100:.2f}%")
print(f"Checkpoint path: {CONFIG['results_dir']}/best_cnn.pth")
print("\nThis saved model is loaded by notebooks 03 and 04.")